In [1]:
import numpy as np, zarr, json, torch, pandas as pd
import dask.array as da
from pathlib import Path
from unet3d import UNet3D
from skimage.feature import peak_local_max
from scipy import ndimage as ndi
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment

VERSION = 'v3'
EPOCHS = '10'
DIST = 6 
THR = 0.8
CKPT = f"models/unet_{VERSION}_e{EPOCHS}.pt" 

VIDEOS = [
    '44b6_0113de3b', 
    '44b6_0b24845f', 
    '6bba_05b6850b', 
    '6bba_05db0fb1'
]

ZARR_FMT = '../test_processed/{}.zarr/0'
GEFF_FMT = "../train/{}.geff"                            
PTS_FMT  = "points/{}_{}_pts.npy"
MASK_FMT = "deadends/cellpose/{}_masks.npy"

In [2]:

def get_geff(path):
    g = zarr.open(path, mode='r')
    
    points = np.column_stack([
        g["nodes/props/t/values"][:],
        g["nodes/props/z/values"][:],
        g["nodes/props/y/values"][:],
        g["nodes/props/x/values"][:],
    ])
    return points

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
nrm = json.load(open(f"models/norm_{VERSION}.json"))
model = UNet3D(16).to(device).eval()
model.load_state_dict(torch.load(CKPT, map_location=device))

Path("points").mkdir(exist_ok=True)
for vid in VIDEOS:
    z = zarr.open(ZARR_FMT.format(vid), mode="r")
    ts = list(range(z.shape[0]))
    
    pts = []
    for t in ts:
        x = np.asarray(z[t]).astype(np.float32)
        x = np.clip((x - nrm["lo"]) / (nrm["hi"] - nrm["lo"] + 1e-8), 0, 1)
    
        with torch.no_grad():
            xb = torch.from_numpy(x)[None, None].to(device)
            with torch.autocast(device_type=device, enabled=(device == "cuda")):
                hm = model(xb)[0, 0].float().cpu().numpy()
    
        c = peak_local_max(hm, min_distance=DIST, threshold_abs=THR, exclude_border=False)  # (n,3) z,y,x
        if len(c):
            pts.append(np.column_stack([np.full(len(c), t), c]))
    
    pts = np.concatenate(pts) if pts else np.zeros((0, 4), int)
    np.save(f"points/{vid}_{VERSION}_pts.npy", pts)
    
    print(vid, pts.shape, "detections over", len(ts), "frames")

44b6_0113de3b (24320, 4) detections over 100 frames
44b6_0b24845f (39901, 4) detections over 100 frames
6bba_05b6850b (6025, 4) detections over 100 frames
6bba_05db0fb1 (63817, 4) detections over 100 frames


In [4]:
PHYS = np.array([1.625, 0.40625, 0.40625])   # µm per voxel (z,y,x)
MATCH_RADIUS = 7.0                           # µm
MIN_VOX = 50                                 # drop tiny cellpose objects

# ---------- matching ----------
def n_matched(dets, ref):
    """One-to-one (Hungarian) matches within MATCH_RADIUS µm."""
    if len(dets) == 0 or len(ref) == 0:
        return 0
    D = cdist(dets * PHYS, ref * PHYS)
    r, c = linear_sum_assignment(D)
    return int((D[r, c] <= MATCH_RADIUS).sum())

def load_gt(geff_path):
    g = zarr.open(geff_path)
    gt = np.column_stack([g[f"nodes/props/{a}/values"][:] for a in ("t", "z", "y", "x")]).astype(float)
    return gt   # (n,4) t,z,y,x in voxels

def load_cellpose_centroids(mask_path, frames):
    masks = np.load(mask_path, mmap_mode="r")
    out = {}
    for t in frames:
        m = np.asarray(masks[int(t)])
        ids, counts = np.unique(m, return_counts=True)
        ids = ids[(ids > 0) & (counts >= MIN_VOX)]
        out[int(t)] = np.array(ndi.center_of_mass(np.ones(m.shape), m, ids)).reshape(-1, 3)
    return out

def match_video(pts, gt, cp_by_frame):
    """Per-frame match counts. pts, gt: (n,4) t,z,y,x. Returns a DataFrame."""
    pts_t, gt_t = pts[:, 0].astype(int), gt[:, 0].astype(int)
    rows = []
    for t in np.unique(gt_t):                       # only frames with GT are scored
        t = int(t)
        G = gt[gt_t == t, 1:]
        D = pts[pts_t == t, 1:].astype(float)
        C = cp_by_frame[t]
        Cx = C[cdist(C * PHYS, G * PHYS).min(1) > MATCH_RADIUS] if len(G) and len(C) else C
        R = np.vstack([G, Cx]) if len(Cx) else G    # reference = GT U unmatched cellpose
        rows.append(dict(t=t, n_gt=len(G), n_cp=len(C), n_ref=len(R), n_det=len(D),
                         hit_gt=n_matched(D, G), hit_cp=n_matched(D, C), hit_ref=n_matched(D, R)))
    return pd.DataFrame(rows)

# ---------- statistics ----------
def f1(p, r):
    return 2 * p * r / max(p + r, 1e-9)

def video_stats(df):
    """Aggregate per-frame counts into one summary dict."""
    S = df.sum(numeric_only=True)
    cp_p, cp_r = S.hit_cp / max(S.n_det, 1), S.hit_cp / max(S.n_cp, 1)
    return dict(
        frames=len(df),
        det_per_frame=S.n_det / len(df),
        n_gt=int(S.n_gt), hit_gt=int(S.hit_gt),
        recall_gt=S.hit_gt / max(S.n_gt, 1),
        n_det=int(S.n_det),
        cp_prec=cp_p, 
        cp_rec=cp_r, 
        cp_f1=f1(cp_p, cp_r)
    )

In [ ]:
summary, per_frame = [], []
for vid in VIDEOS:
    pts = np.load(PTS_FMT.format(vid, VERSION))
    gt = load_gt(GEFF_FMT.format(vid))
    cp = load_cellpose_centroids(MASK_FMT.format(vid), np.unique(gt[:, 0].astype(int)))

    df = match_video(pts, gt, cp)
    df.insert(0, "video", vid)
    per_frame.append(df)
    summary.append(dict(video=vid, **video_stats(df)))
    print("done", vid)

per_frame = pd.concat(per_frame, ignore_index=True)
summary = pd.DataFrame(summary)

# overall = pooled over all videos (sums the counts, not an average of ratios)
tot = video_stats(per_frame)
summary = pd.concat([summary, pd.DataFrame([dict(video="ALL (pooled)", **tot)])], ignore_index=True)

pd.set_option("display.float_format", "{:.3f}".format)
print(summary.to_string(index=False))
per_frame.to_csv(F"points/eval_per_frame_{VERSION}.csv", index=False)
summary.to_csv(F"points/eval_summary_{VERSION}.csv", index=False)

done 44b6_0113de3b
done 44b6_0b24845f
done 6bba_05b6850b
done 6bba_05db0fb1
        video  frames  det_per_frame  n_gt  hit_gt  recall_gt  n_det  cp_prec  cp_rec  cp_f1
44b6_0113de3b      52        251.269    52      52      1.000  13066    0.913   0.900  0.906
44b6_0b24845f      40        335.050    51      44      0.863  13402    0.083   0.666  0.148
6bba_05b6850b     100         60.250   861     799      0.928   6025    0.651   0.678  0.664
6bba_05db0fb1     100        638.170  1229    1102      0.897  63817    0.738   0.762  0.750
 ALL (pooled)     292        329.829  2193    1997      0.911  96310    0.665   0.776  0.717


: 

results from model v2

| video | frames | det\_per\_frame | n\_gt | hit\_gt | recall\_gt | n\_det | cp\_prec | cp\_rec | cp\_f1 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 44b6\_0113de3b | 52 | 263.538 | 52 | 52 | 1.000 | 13704 | 0.901 | 0.932 | 0.916 |
| 44b6\_0b24845f | 40 | 418.600 | 51 | 47 | 0.922 | 16744 | 0.071 | 0.714 | 0.130 |
| 6bba\_05b6850b | 100 | 71.340 | 861 | 831 | 0.965 | 7134 | 0.666 | 0.820 | 0.735 |
| 6bba\_05db0fb1 | 100 | 682.500 | 1229 | 1113 | 0.906 | 68250 | 0.730 | 0.806 | 0.766 |
| ALL (pooled) | 292 | 362.438 | 2193 | 2043 | 0.932 | 105832 | 0.644 | 0.825 | 0.723 |

results from model v1

| video | frames | det\_per\_frame | n\_gt | hit\_gt | recall\_gt | n\_det | cp\_prec | cp\_rec | cp\_f1 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 44b6\_0113de3b | 52 | 265.346 | 52 | 52 | 1.000 | 13798 | 0.898 | 0.935 | 0.916 |
| 44b6\_0b24845f | 40 | 423.300 | 51 | 47 | 0.922 | 16932 | 0.071 | 0.719 | 0.130 |
| 6bba\_05b6850b | 100 | 74.430 | 861 | 809 | 0.940 | 7443 | 0.652 | 0.838 | 0.734 |
| 6bba\_05db0fb1 | 100 | 680.650 | 1229 | 1096 | 0.892 | 68065 | 0.725 | 0.798 | 0.760 |
| ALL (pooled) | 292 | 363.829 | 2193 | 2004 | 0.914 | 106238 | 0.638 | 0.821 | 0.718 |

inferenced model v0 on preprocessed volumes

| video | frames | det\_per\_frame | n\_gt | hit\_gt | recall\_gt | n\_det | cp\_prec | cp\_rec | cp\_f1 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 44b6\_0113de3b | 52 | 261.577 | 52 | 52 | 1.000 | 13602 | 0.907 | 0.931 | 0.919 |
| 44b6\_0b24845f | 40 | 397.550 | 51 | 47 | 0.922 | 15902 | 0.074 | 0.700 | 0.133 |
| 6bba\_05b6850b | 100 | 66.650 | 861 | 788 | 0.915 | 6665 | 0.648 | 0.745 | 0.693 |
| 6bba\_05db0fb1 | 100 | 664.310 | 1229 | 1107 | 0.901 | 66431 | 0.736 | 0.791 | 0.762 |
| ALL (pooled) | 292 | 351.370 | 2193 | 1994 | 0.909 | 102600 | 0.650 | 0.808 | 0.721 |

inferenced model v0 on non-preprocessed volumes

| Video         | Frames | Det/frame | n_gt  | hit_gt | recall_gt | n_det  | cp_prec | cp_rec | cp_f1 |
| ------------- | ------ | --------- | ----- | ------ | --------- | ------ | ------- | ------ | ----- |
| 44b6_0113de3b | 52     | 273.404   | 52    | 52     | 1.000     | 14,217 | 0.885   | 0.949  | 0.916 |
| 44b6_0b24845f | 40     | 170.675   | 51    | 11     | 0.216     | 6,827  | 0.118   | 0.481  | 0.190 |
| 6bba_05b6850b | 100    | 64.170    | 861   | 602    | 0.699     | 6,417  | 0.643   | 0.713  | 0.676 |
| 6bba_05db0fb1 | 100    | 630.520   | 1,229 | 1,074  | 0.874     | 63,052 | 0.736   | 0.750  | 0.743 |
| ALL (pooled)  | 292    | 309.976   | 2,193 | 1,739  | 0.793     | 90,513 | 0.706   | 0.774  | 0.739 |

In [ ]:
import napari, numpy as np
SCALE = (1.0, 1.625, 0.40625, 0.40625)

for vid in VIDEOS[2:3]:
    video = da.from_zarr(f'../test/{vid}.zarr/0')
    proc = da.from_zarr(f'../test_processed/{vid}.zarr/0')
    pts = get_geff(f"../train/{vid}.geff")
    labels = np.load(f'points/{vid}_{VERSION}_pts.npy')

    viewer = napari.Viewer(ndisplay=3, title=vid)
    try:
        viewer.axes.visible = viewer.axes.colored = viewer.axes.labels = True
    except AttributeError:
        a = viewer.scene.overlays.axes
        a.visible = a.colored = a.labels = True
    
    viewer.add_image(
        video, 
        name='orig_vol',
        scale=SCALE, 
        colormap='magma', 
        blending='additive'
    )

    viewer.add_image(
        proc, 
        name='proc_vol',
        scale=SCALE, 
        colormap='magma', 
        blending='additive'
    )
    
    gt_layer = viewer.add_points(
        pts,
        name='gt',
        scale=SCALE,
        face_color='cyan',
        size=4,
    )

    gt_layer_r = viewer.add_points(
        pts,
        name="gt_r",
        scale=SCALE,
        face_color="cyan",
        size=14.0,
        opacity=0.3,
    )

    pred_layer = viewer.add_points(
        labels,
        name='pred',
        scale=SCALE,
        face_color='brown',
        size=4,
    )

    @viewer.bind_key('g')
    def _goto(v):
        sel = list(gt_layer.selected_data)
        if not sel:
            v.status = 'Select a GT point first'
            return

        t, z, y, x = gt_layer.data[sel[0]] * SCALE

        v.dims.ndisplay = 2
        v.dims.set_point(0, t)
        v.dims.set_point(1, z)
        v.camera.center = (0, y, x)
        v.camera.zoom = 6

    napari.run()
